In [2]:
!pip install -q bert-score

import os
import pandas as pd
from bert_score import score

# 1. Automatically locate prediction files
mimic_path, iu_path = None, None
for root, _, files in os.walk('/kaggle/input'):
  for f in files:
    if f == 'model7_test_predictions.csv':
      mimic_path = os.path.join(root, f)
    elif f == 'model7_ood_predictions.csv':
      iu_path = os.path.join(root, f)

print(f'MIMIC File: {mimic_path}')
print(f'IU File   : {iu_path}')


def clean_text_list(series):
  return [str(s).strip() if pd.notna(s) and str(s).strip() else '.' for s in series]


# 2. Compute BERTScore on MIMIC-CXR test set (6,388 samples)
print('\nCalculating BERTScore for MIMIC-CXR (6,388 samples)...')
df_mimic = pd.read_csv(mimic_path)
cands_m = clean_text_list(df_mimic['prediction'])
refs_m = clean_text_list(df_mimic['ground_truth'])

_, _, F1_m = score(
    cands_m,
    refs_m,
    model_type='bert-base-uncased',
    device='cuda',
    batch_size=64,
    verbose=False,
)
mimic_bert_f1 = F1_m.mean().item()

# 3. Compute BERTScore on IU X-Ray OOD cohort (3,666 samples)
print('Calculating BERTScore for IU X-Ray (3,666 samples)...')
df_iu = pd.read_csv(iu_path)
cands_iu = clean_text_list(df_iu['prediction'])
refs_iu = clean_text_list(df_iu['ground_truth'])

_, _, F1_iu = score(
    cands_iu,
    refs_iu,
    model_type='bert-base-uncased',
    device='cuda',
    batch_size=64,
    verbose=False,
)
iu_bert_f1 = F1_iu.mean().item()

# 4. Display benchmark results
print('\n' + '=' * 55)
print(f'MIMIC-CXR (IID) BERTScore F1 : {mimic_bert_f1:.4f}')
print(f'IU X-Ray  (OOD) BERTScore F1 : {iu_bert_f1:.4f}')
print('=' * 55)

MIMIC File: /kaggle/input/notebooks/mikuchinu/mimic-cxr-model-7-eval/model7_test_predictions.csv
IU File   : /kaggle/input/notebooks/mikuchinu/mimic-cxr-model-7-ood-eval/model7_ood_predictions.csv

Calculating BERTScore for MIMIC-CXR (6,388 samples)...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (2 outputs left out)

Calculating BERTScore for IU X-Ray (3,666 samples)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



MIMIC-CXR (IID) BERTScore F1 : 0.6562
IU X-Ray  (OOD) BERTScore F1 : 0.6844
